In [ ]:
import os
import sys
import json
import json
from tqdm import tqdm

In [ ]:
Path = "/home/user_name/U-MARVEL"
# 训练集候选池路径
train_cand_path = os.path.join(Path, "data/M-BEIR/cand_pool/global/mbeir_union_train_cand_pool.jsonl")
# 测试集候选池路径
union_test_cand_pool_path = os.path.join(Path,"data/M-BEIR/cand_pool/global/mbeir_union_test_cand_pool.jsonl")

In [ ]:
with open(train_cand_path, 'r', encoding='utf-8') as f:
    total_lines = sum(1 for _ in f)
print(total_lines)
with open(union_test_cand_pool_path, 'r', encoding='utf-8') as f:
    total_lines = sum(1 for _ in f)
print(total_lines)

In [ ]:
file_names = [
    "mbeir_cirr_task7_test.jsonl",
    "mbeir_edis_task2_test.jsonl",
    "mbeir_fashion200k_task0_test.jsonl",
    "mbeir_fashion200k_task3_test.jsonl",
    "mbeir_fashioniq_task7_test.jsonl",
    "mbeir_infoseek_task6_test.jsonl",
    "mbeir_infoseek_task8_test.jsonl",
    "mbeir_mscoco_task0_test.jsonl",
    "mbeir_mscoco_task3_test.jsonl",
    "mbeir_nights_task4_test.jsonl",
    "mbeir_oven_task6_test.jsonl",
    "mbeir_oven_task8_test.jsonl",
    "mbeir_visualnews_task0_test.jsonl",
    "mbeir_visualnews_task3_test.jsonl",
    "mbeir_webqa_task1_test.jsonl",
    "mbeir_webqa_task2_test.jsonl"
]

print(len(file_names))
total_lines = 0
Path_temp = "/home/user_name/U-MARVEL/data/M-BEIR/query/test"
for file_name in file_names:
    file_name = os.path.join(Path_temp,file_name)
    try:
        with open(file_name, 'r', encoding='utf-8') as file:
            lines = file.readlines()
            line_count = len(lines)
            total_lines += line_count
            print(f"{file_name} 的行数: {line_count}")
    except FileNotFoundError:
        print(f"错误: 文件 {file_name} 未找到。")
    except Exception as e:
        print(f"错误: 读取文件 {file_name} 时发生未知错误: {e}")

print(f"所有文件的总行数: {total_lines}")

In [ ]:
query_union_train = "/home/user_name/U-MARVEL/data/M-BEIR/query/union_train/mbeir_union_up_train.jsonl"
with open(query_union_train, 'r', encoding='utf-8') as file:
    lines = file.readlines()
print(f"训练集所有文件的总行数: {len(lines)}")
print(f"测试集所有文件的总行数: {total_lines}")

In [ ]:
# 加载 cand 集合， did 是键，value 是 数据
def load_jsonl(file_path):
    """
    Load a JSONL file into a list of dictionaries with a progress bar.
    """
    # 第一次遍历文件以统计总行数（用于进度条）
    with open(file_path, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for _ in f)
    
    # 第二次遍历文件并解析数据
    data = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, total=total_lines, desc="Loading JSONL"):
            item = json.loads(line)
            data[item['did']] = item.copy()
    print(f"Loaded {len(data)} records from {file_path}")
    return data

In [ ]:
# 加载 query 集合，qid 是键，value 是正负样本的 did 合并列表
def load_jsonl_query(file_path):
    """
    Load a JSONL file into a list of dictionaries with a progress bar.
    """
    # 第一次遍历文件以统计总行数（用于进度条）
    with open(file_path, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for _ in f)
    
    # 第二次遍历文件并解析数据
    data = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, total=total_lines, desc="Loading JSONL"):
            item = json.loads(line)
            pos_cand_list = item["pos_cand_list"]
            neg_cand_list = item["neg_cand_list"]
            qid = item["qid"]
            data[qid] = pos_cand_list[:] + neg_cand_list[:]
    print(f"Loaded {len(data)} records from {file_path}")
    return data

### 检查训练集的候选池和测试集候选池互相的包含情况

In [ ]:
# Load the training candidate pool
train_cand_pool = load_jsonl(train_cand_path)

In [ ]:
union_test_cand_pool = load_jsonl(union_test_cand_pool_path)
print(f"Union test candidate pool size: {len(union_test_cand_pool)}")
# Union test candidate pool size: 5609079

In [ ]:
# 检验 train_cand_pool 当中的数据是否在 union_test_cand_pool 当中
def check_candidates_in_union_test(train_cand_pool, union_test_cand_pool):
    """
    Check if all candidates in the training candidate pool are present in the union test candidate pool.
    """
    not_in_union_test = []
    for key in tqdm(train_cand_pool.keys(), desc="Checking candidates"):
        if key not in union_test_cand_pool:
            not_in_union_test.append(key)
    
    print(f"Number of candidates not in union test: {len(not_in_union_test)}")
    return not_in_union_test
# Check candidates in union test
not_in_union_test = check_candidates_in_union_test(train_cand_pool, union_test_cand_pool)
print(len(set(not_in_union_test)))


# Number of candidates not in union test: 614353
# 614353

In [ ]:
not_in_union_test[:10]

### 检查训练集当中非 union 的 query 的 qid 以及正负样本的 did 是否在 union 候选池当中

In [ ]:
query_train_list = [
    "mbeir_cirr_train.jsonl",
    "mbeir_edis_train.jsonl",
    "mbeir_fashion200k_train.jsonl",
    "mbeir_fashioniq_train.jsonl",
    "mbeir_infoseek_train.jsonl",
    "mbeir_mscoco_train.jsonl",
    "mbeir_nights_train.jsonl",
    "mbeir_oven_train.jsonl",
    "mbeir_visualnews_train.jsonl",
    "mbeir_webqa_train.jsonl"
]
Path_query_train = "/home/user_name/U-MARVEL/data/M-BEIR/query/train"
query_train_list = [os.path.join(Path_query_train, path) for path in query_train_list]
print("query_train_list:", query_train_list)

In [ ]:
# 收集 query_train_list 当中所有的 pos_cand_list, neg_cand_list, qid
query_train_pos_cand_list = []
query_train_qid_list = []
query_train_neg_cand_list = []
for query_train_path in query_train_list:
    with open(query_train_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc=f"Loading {query_train_path}"):
            item = json.loads(line)
            qid = item['qid']
            pos_cand_list = item['pos_cand_list']
            neg_cand_list = item["neg_cand_list"]
            query_train_qid_list.append(qid)
            query_train_pos_cand_list.extend(pos_cand_list)
            query_train_neg_cand_list.extend(neg_cand_list)
print(f"Total number of pos candidates in query train list: {len(query_train_pos_cand_list)} {len(set(query_train_pos_cand_list))}")
print(f"Total number of neg candidates in query train list: {len(query_train_neg_cand_list)} {len(set(query_train_neg_cand_list))}")
print(f"Total number of qid in query train list: {len(query_train_qid_list)} {len(set(query_train_qid_list))}")

In [ ]:
# 收集 train_cand_path 当中所有的 did
train_cand_did_list = []
train_cand_pool = load_jsonl(train_cand_path)
train_cand_did_list = list(train_cand_pool.keys())
print(f"Total number of candidates in train cand pool: {len(train_cand_did_list)} {len(set(train_cand_did_list))}")
query_train_pos_cand_set = set(query_train_pos_cand_list)
query_train_neg_cand_set = set(query_train_neg_cand_list)
train_cand_did_set = set(train_cand_did_list)

#### 检查 query 当中的正样本 did 是否在候选池当中

In [ ]:
# 交集
intersection = query_train_pos_cand_set.intersection(train_cand_did_set)
print(f"Number of candidates in both query train list and train cand pool: {len(intersection)}")
# 差集，即在 query_train_pos_cand_set 中但不在 train_cand_did_set 中的元素
difference = query_train_pos_cand_set.difference(train_cand_did_set)
print(f"Number of candidates in query train list but not in train cand pool: {len(difference)}")
# 差集，即在 train_cand_did_set 中但不在 query_train_pos_cand_set 中的元素
difference = train_cand_did_set.difference(query_train_pos_cand_set)
print(f"Number of candidates in train cand pool but not in query train list: {len(difference)}")

#### 检查 query 当中的负样本 did 是否在候选池当中

In [ ]:
# 交集
intersection = query_train_neg_cand_set.intersection(train_cand_did_set)
print(f"Number of candidates in both query train list and train cand pool: {len(intersection)}")
# 差集，即在 query_train_neg_cand_set 中但不在 train_cand_did_set 中的元素
difference = query_train_neg_cand_set.difference(train_cand_did_set)
print(f"Number of candidates in query train list but not in train cand pool: {len(difference)}")
# 差集，即在 train_cand_did_set 中但不在 query_train_neg_cand_set 中的元素
difference = train_cand_did_set.difference(query_train_neg_cand_set)
print(f"Number of candidates in train cand pool but not in query train list: {len(difference)}")

#### 检验非 union 集合的 qid 和 union 集合的 qid 情况

In [ ]:
query_union_train = "/home/user_name/U-MARVEL/data/M-BEIR/query/union_train/mbeir_union_up_train.jsonl"
with open(query_union_train, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc=f"Loading {query_train_path}"):
        item = json.loads(line)
        print(item)
        break

In [ ]:
# 收集 query_union_train 当中所有的 qid
query_union_train_qid_ist = []
with open(query_union_train, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc=f"Loading {query_union_train}"):
        item = json.loads(line)
        qid = item['qid']
        query_union_train_qid_ist.append(qid)
print(f"Total number of qid in query union train list: {len(query_union_train_qid_ist)} {len(set(query_union_train_qid_ist))}")

In [ ]:
# 检验 query_train_qid_list 是否在 query_union_train_qid_ist 当中
query_train_qid_set = set(query_train_qid_list)
query_union_train_qid_set = set(query_union_train_qid_ist)
# 交集
intersection = query_train_qid_set.intersection(query_union_train_qid_set)
print(f"Number of qid in both query train list and query union train list: {len(intersection)}")
# 差集，即在 query_train_qid_set 中但不在 query_union_train_qid_set 中的元素
difference = query_train_qid_set.difference(query_union_train_qid_set)
print(f"Number of qid in query train list but not in query union train list: {len(difference)}")
# 差集，即在 query_union_train_qid_set 中但不在 query_train_qid_set 中的元素
difference = query_union_train_qid_set.difference(query_train_qid_set)
print(f"Number of qid in query union train list but not in query train list: {len(difference)}")
# 打印 query_union_train_qid_ist 当中重复的 qid
from collections import Counter
from collections import defaultdict
qid_counts = Counter(query_union_train_qid_ist)
duplicates = [qid for qid, count in qid_counts.items() if count > 1]
print(f"Number of duplicate qid in query union train list: {len(duplicates)}")
print(f"Duplicate qid in query union train list: {duplicates[:10]}")

### 对训练集进行拆分

In [ ]:
import os
import sys
import json
from tqdm import tqdm
from collections import defaultdict
Path_query_train = "/home/user_name/U-MARVEL/data/M-BEIR/query/train/query_train"
# 任务映射
dataset_to_val_data_file_middle_name_map = {
            "VisualNews": ["visualnews_task0", "visualnews_task3"],
            "MSCOCO": ["mscoco_task0", "mscoco_task3"],
            "Fashion200K": ["fashion200k_task0", "fashion200k_task3"],
            "WebQA": ["webqa_task1", "webqa_task2"],
            "EDIS": ["edis_task2"],
            "NIGHTS": ["nights_task4"],
            "OVEN": ["oven_task6", "oven_task8"],
            "INFOSEEK": ["infoseek_task6", "infoseek_task8"],
            "FashionIQ": ["fashioniq_task7"],
            "CIRR": ["cirr_task7"],
        }

# Mapping of dataset names to IDs
DATASET_IDS = {
    "visualnews": 0,     "fashion200k": 1,   "webqa": 2,      "edis": 3,
    "nights": 4,         "oven": 5,          "infoseek": 6,
    "fashioniq": 7,      "cirr": 8,          "mscoco": 9,
}

MBEIR_TASK = {
    "text -> image": 0,      "text -> text": 1,             "text -> image,text": 2,
    "image -> text": 3,      "image -> image": 4,           "image -> text,image": 5,
    "image,text -> text": 6, "image,text -> image": 7,      "image,text -> image,text": 8,
}
# "./data/M-BEIR/query/train/mbeir_train_visualnews_task3.jsonl",
query_train_file_list = []
for dataset,datatset_task_name in dataset_to_val_data_file_middle_name_map.items():
    for task_name in datatset_task_name:
        query_train_file_list.append(os.path.join(Path_query_train, f"mbeir_train_{task_name}.jsonl"))
print("query_train_file_list[0]:", query_train_file_list[0])
print(len(query_train_file_list))
query_train_file_list2datasetidtaskid = {}

In [ ]:
for query_train_file in query_train_file_list:
    task_id = int(query_train_file.split("/")[-1].split("_")[-1].split(".")[0][-1])
    datasetid = DATASET_IDS[query_train_file.split("/")[-1].split("_")[-2]]
    query_train_file_list2datasetidtaskid[(datasetid, task_id)] = query_train_file
print(query_train_file_list2datasetidtaskid)

In [ ]:
# 读取数据
datasetidtaskid2querylist = defaultdict(list)
query_union_train = "/home/user_name/U-MARVEL/data/M-BEIR/query/union_train/mbeir_union_up_train.jsonl"
with open(query_union_train, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc=f"Loading {query_union_train}"):
        item = json.loads(line)
        datasetid = int(item['qid'].split(":")[0])
        task_id = int(item["task_id"])
        datasetidtaskid2querylist[(datasetid, task_id)].append(item)
print(f"Total number of datasetidtaskid2querylist: {len(datasetidtaskid2querylist)}")
# 打印 datasetidtaskid2querylist 当中的数据
for key, value in datasetidtaskid2querylist.items():
    print(f"Key: {key}, Value: {len(value)}")

In [ ]:
# 根据 datasetidtaskid2querylist 当中的数据和 query_train_file_list2datasetidtaskid 的文件路径，生成新的jsonl文件
for key, value in datasetidtaskid2querylist.items():
    datasetid, task_id = key
    query_train_file = query_train_file_list2datasetidtaskid[key]
    print(f"Key: {key}, Value: {len(value)}, query_train_file: {query_train_file}")
    # 将 value 当中的数据写入 query_train_file 当中
    with open(query_train_file, 'w', encoding='utf-8') as f:
        for item in value:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    # print(f"Write {len(value)} records to {query_train_file}")

In [ ]:
# 读取 query_train_file_list2datasetidtaskid 文件，获取 pos_cand_list 和 neg_cand_list
query_train_file_list2pos_cand_list = defaultdict(set)
query_train_file_list2neg_cand_list = defaultdict(set)
query_train_file_list2datasetid = defaultdict(set)
for key,query_train_file in query_train_file_list2datasetidtaskid.items():
    with open(query_train_file, 'r', encoding='utf-8') as f:
        for line in tqdm(f):
            item = json.loads(line)
            pos_cand_list = item['pos_cand_list']
            neg_cand_list = item['neg_cand_list']
            query_train_file_list2pos_cand_list[query_train_file].update(set(pos_cand_list))
            query_train_file_list2neg_cand_list[query_train_file].update(set(neg_cand_list))
            for did in pos_cand_list+neg_cand_list:
                datasetid = int(did.split(":")[0])
                query_train_file_list2datasetid[query_train_file].add(datasetid)
    print(f"Total number of datasetid in {query_train_file}: {(query_train_file_list2datasetid[query_train_file])}")
print(query_train_file_list2datasetid)

In [ ]:
from tqdm import tqdm

# 检查 query_train_file_list2pos_cand_list 和 query_train_file_list2neg_cand_list 的 modality 是否一致
train_cand_pool = load_jsonl(train_cand_path)

# 检查正样本候选列表
print("Checking positive candidates...")
for query_train_file, pos_cand_set in query_train_file_list2pos_cand_list.items():
    modality = None
    print(query_train_file,"正样本数量: ",len(pos_cand_set))
    for did in tqdm(pos_cand_set):
        if modality is None:
            modality = train_cand_pool[did]["modality"]
        else:
            if modality != train_cand_pool[did]["modality"]:
                print(f"\nInconsistency found in positive candidates for query: {query_train_file}")
                print(f"Modality: {modality}, {train_cand_pool[did]['modality']}")
                print(train_cand_pool[did])
                break
    neg_modality_num = 0
    neg_cand_set = query_train_file_list2neg_cand_list[query_train_file]
    print(query_train_file,"负样本数量: ",len(neg_cand_set))
    for did in tqdm(neg_cand_set):
        if modality is None:
            modality = train_cand_pool[did]["modality"]
        else:
            if modality != train_cand_pool[did]["modality"]:
                neg_modality_num +=1
                # print(f"\nInconsistency found in negative candidates for query: {query_train_file}")
                # print(f"Modality: {modality}, {train_cand_pool[did]['modality']}")
                # print(train_cand_pool[did])
                # break
    print("负样本不匹配模态的数量: ",neg_modality_num)

In [ ]:
# 检查负样本候选列表
print("\nChecking negative candidates...")
for query_train_file, neg_cand_set in query_train_file_list2neg_cand_list.items():
    modality = None
    neg_modality_num = 0
    print(query_train_file,len(neg_cand_set))
    for did in tqdm(neg_cand_set):
        if modality is None:
            modality = train_cand_pool[did]["modality"]
        else:
            if modality != train_cand_pool[did]["modality"]:
                neg_modality_num +=1
                print(f"\nInconsistency found in negative candidates for query: {query_train_file}")
                print(f"Modality: {modality}, {train_cand_pool[did]['modality']}")
                print(train_cand_pool[did])
                break
    print("不匹配模态的数量: ",neg_modality_num)

### 对 cand 进行拆分

In [ ]:
import os
import sys
import json
from tqdm import tqdm
from collections import defaultdict
Path_cand_train = "/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train"
# 任务映射
dataset_to_val_data_file_middle_name_map = {
            "VisualNews": ["visualnews_task0", "visualnews_task3"],
            "MSCOCO": ["mscoco_task0", "mscoco_task3"],
            "Fashion200K": ["fashion200k_task0", "fashion200k_task3"],
            "WebQA": ["webqa_task1", "webqa_task2"],
            "EDIS": ["edis_task2"],
            "NIGHTS": ["nights_task4"],
            "OVEN": ["oven_task6", "oven_task8"],
            "INFOSEEK": ["infoseek_task6", "infoseek_task8"],
            "FashionIQ": ["fashioniq_task7"],
            "CIRR": ["cirr_task7"],
        }

# Mapping of dataset names to IDs
DATASET_IDS = {
    "visualnews": 0,     "fashion200k": 1,   "webqa": 2,      "edis": 3,
    "nights": 4,         "oven": 5,          "infoseek": 6,
    "fashioniq": 7,      "cirr": 8,          "mscoco": 9,
}

MBEIR_TASK = {
    0: "text -> image",      1: "text -> text",             2: "text -> image,text",
    3: "image -> text",      4: "image -> image",           5:"image -> text,image",
    6: "image,text -> text", 7:"image,text -> image",       8:"image,text -> image,text",
}
# "./data/M-BEIR/query/train/mbeir_cirr_task7_cand_pool.jsonl",
cand_train_file_list = []
for dataset,datatset_task_name in dataset_to_val_data_file_middle_name_map.items():
    for task_name in datatset_task_name:
        cand_train_file_list.append(os.path.join(Path_cand_train, f"mbeir_train_{task_name}_cand_pool.jsonl"))
print("cand_train_file_list[0]:", cand_train_file_list[0])
print(len(cand_train_file_list))
print(cand_train_file_list)

In [ ]:
cand_train_file_list

In [ ]:
cand_train_file_list2datasetid = defaultdict(set)
for cand_train_file in cand_train_file_list:
    task_id = int(cand_train_file.split("/")[-1].split("_")[-3].split(".")[0][-1])
    datasetid = DATASET_IDS[cand_train_file.split("/")[-1].split("_")[-4]]
    cand_train_file_list2datasetid[cand_train_file] = {datasetid}

In [ ]:
print(cand_train_file_list2datasetid)

In [ ]:
cand_train_file_list2datasetid = {
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_visualnews_task0_cand_pool.jsonl': {0}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_visualnews_task3_cand_pool.jsonl': {0}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_mscoco_task0_cand_pool.jsonl': {9}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_mscoco_task3_cand_pool.jsonl': {9}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_fashion200k_task0_cand_pool.jsonl': {1}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_fashion200k_task3_cand_pool.jsonl': {1}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_webqa_task1_cand_pool.jsonl': {2}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_webqa_task2_cand_pool.jsonl': {2}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_edis_task2_cand_pool.jsonl': {3}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_nights_task4_cand_pool.jsonl': {4}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_oven_task6_cand_pool.jsonl': {5,6}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_oven_task8_cand_pool.jsonl': {5,6}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_infoseek_task6_cand_pool.jsonl': {5,6}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_infoseek_task8_cand_pool.jsonl': {5,6}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_fashioniq_task7_cand_pool.jsonl': {7}, 
    '/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_train/mbeir_train_cirr_task7_cand_pool.jsonl': {8}
}

In [ ]:
Path = "/home/user_name/U-MARVEL"
train_cand_path = os.path.join(Path, "data/M-BEIR/cand_pool/global/mbeir_union_train_cand_pool.jsonl")
train_cand_pool = {}
datasetid2did = defaultdict(set)
with open(train_cand_path, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        item = json.loads(line)
        did = item['did']
        datasetid = int(did.split(":")[0])
        train_cand_pool[did] = item.copy()
        datasetid2did[datasetid].add(did)
    
print(f"Total number of datasetid in train cand pool: {len(datasetid2did)}")
print(f"Total number of did in train cand pool: {len(train_cand_pool)}")

In [ ]:
num = 0
for datasetid,did in datasetid2did.items():
    num += len(did)
    print(datasetid,len(did))
print(num)

In [ ]:
# 根据 cand_train_file_list2datasetid，datasetid2did， train_cand_pool 生成新的jsonl文件
from tqdm import tqdm
for cand_train_file,datasetid in cand_train_file_list2datasetid.items():
    # 将 datasetid 当中的数据写入 cand_train_file 当中
    task_id = int(cand_train_file.split("/")[-1].split("_")[-3].split(".")[0][-1])
    task_name = MBEIR_TASK[task_id]
    task_modality = task_name.split(" -> ")[1]
    with open(cand_train_file, 'w', encoding='utf-8') as f:
        for item in datasetid:
            num = 0
            for did in tqdm(list(datasetid2did[item])):
                if did in train_cand_pool:
                    item_data = train_cand_pool[did]
                    modality = item_data["modality"]
                    if modality == task_modality:
                        f.write(json.dumps(train_cand_pool[did], ensure_ascii=False) + '\n')
                        num += 1
                else:
                    print(f"did: {did} not in train_cand_pool")
            print(task_modality)
            print(f"cand_train_file: {cand_train_file}, datasetid: {datasetid}")
            print(f"Write {num} of {len(datasetid2did[item])} records to {cand_train_file}")

### 检验 16 个文件的正负样本 did 都在对应的候选池当中

In [ ]:
# 建立 query_train_file_list 和 cand_train_file_list 的映射关系
query_train_file_list2cand_train_file_list = {}
for query_train_file in query_train_file_list:
    dataset = query_train_file.split("/")[-1].split("_")[2]
    task = query_train_file.split("/")[-1].split("_")[3].split(".")[0]
    # print(query_train_file,dataset,task)
    for cand_train_file in cand_train_file_list:
        # print(cand_train_file)
        if dataset in cand_train_file and task in cand_train_file:
            query_train_file_list2cand_train_file_list[query_train_file] = cand_train_file
            break
print(query_train_file_list2cand_train_file_list)

In [ ]:
# 根据映射关系读取文件，判断 query_train_file_list 当中的 pos_cand_list 和 neg_cand_list 是否在 cand_train_file_list 当中
for query_train_file,cand_train_file in query_train_file_list2cand_train_file_list.items():
    print(f"query_train_file: {query_train_file}, cand_train_file: {cand_train_file}")
    # 读取 cand_train_file
    cand_train_did_set = set()
    with open(cand_train_file, 'r', encoding='utf-8') as f:
        for line in tqdm(f):
            item = json.loads(line)
            did = item['did']
            cand_train_did_set.add(did)
    print(f"Total number of did in {cand_train_file}: {len(cand_train_did_set)}")
    # 读取 query_train_file
    with open(query_train_file, 'r', encoding='utf-8') as f:
        for line in tqdm(f):
            item = json.loads(line)
            pos_cand_list = item['pos_cand_list']
            neg_cand_list = item['neg_cand_list']
            # 检查 pos_cand_list 和 neg_cand_list 是否在 cand_train_did_set 当中
            for did in pos_cand_list:
                if did not in cand_train_did_set:
                    print(f"did: {did} not in cand_train_did_set")
            # for did in neg_cand_list:
            #     if did not in cand_train_did_set:
            #         print(f"did: {did} not in cand_train_did_set")

### 检查测试集候选池的模态是否全部一致

In [ ]:
import os
import json
from collections import defaultdict

In [ ]:

test_cand_pool_files = [
    "mbeir_cirr_task7_cand_pool.jsonl",
    "mbeir_edis_task2_cand_pool.jsonl",
    "mbeir_fashion200k_task0_cand_pool.jsonl",
    "mbeir_fashion200k_task3_cand_pool.jsonl",
    "mbeir_fashioniq_task7_cand_pool.jsonl",
    "mbeir_infoseek_task6_cand_pool.jsonl",
    "mbeir_infoseek_task8_cand_pool.jsonl",
    "mbeir_mscoco_task0_test_cand_pool.jsonl",
    "mbeir_mscoco_task0_val_cand_pool.jsonl",
    "mbeir_mscoco_task3_test_cand_pool.jsonl",
    "mbeir_mscoco_task3_val_cand_pool.jsonl",
    "mbeir_nights_task4_cand_pool.jsonl",
    "mbeir_oven_task6_cand_pool.jsonl",
    "mbeir_oven_task8_cand_pool.jsonl",
    "mbeir_visualnews_task0_cand_pool.jsonl",
    "mbeir_visualnews_task3_cand_pool.jsonl",
    "mbeir_webqa_task1_cand_pool.jsonl",
    "mbeir_webqa_task2_cand_pool.jsonl"
]
Path_cand_test = "/home/user_name/U-MARVEL/data/M-BEIR/cand_pool/local/cand_test"
test_cand_pool_files = [os.path.join(Path_cand_test, path) for path in test_cand_pool_files]

In [ ]:
import json
from collections import Counter
from tqdm import tqdm
from collections import defaultdict

def count_modalities(jsonl_path):
    # modality_counter = Counter()
    modality_counter = defaultdict(int)
    with open(jsonl_path, 'r') as file:
        for line in tqdm(file):
            try:
                data = json.loads(line.strip())
                modality = data.get('modality')
                if modality:
                    modality_counter[modality] += 1
            except json.JSONDecodeError:
                print(f"警告：无法解析行: {line}")
    return modality_counter
for jsonl_path in test_cand_pool_files:
    modality_counts = count_modalities(jsonl_path)
    print(jsonl_path,"Modality 统计结果:")
    for mod, count in modality_counts.items():
        print(f"- {mod}: {count}")

### 拆分 qrels 数据集

In [ ]:
import os
import sys
import json
from tqdm import tqdm
from collections import defaultdict
Path_qrels_train = "/home/user_name/U-MARVEL/data/M-BEIR/qrels/train"
# 任务映射
dataset_to_val_data_file_middle_name_map = {
            "VisualNews": ["visualnews_task0", "visualnews_task3"],
            "MSCOCO": ["mscoco_task0", "mscoco_task3"],
            "Fashion200K": ["fashion200k_task0", "fashion200k_task3"],
            "WebQA": ["webqa_task1", "webqa_task2"],
            "EDIS": ["edis_task2"],
            "NIGHTS": ["nights_task4"],
            "OVEN": ["oven_task6", "oven_task8"],
            "INFOSEEK": ["infoseek_task6", "infoseek_task8"],
            "FashionIQ": ["fashioniq_task7"],
            "CIRR": ["cirr_task7"],
        }

# Mapping of dataset names to IDs
DATASET_IDS = {
    "visualnews": 0,     "fashion200k": 1,   "webqa": 2,      "edis": 3,
    "nights": 4,         "oven": 5,          "infoseek": 6,
    "fashioniq": 7,      "cirr": 8,          "mscoco": 9,
}

MBEIR_TASK = {
    0: "text -> image",      1: "text -> text",             2: "text -> image,text",
    3: "image -> text",      4: "image -> image",           5:"image -> text,image",
    6: "image,text -> text", 7:"image,text -> image",       8:"image,text -> image,text",
}
# "./data/M-BEIR/query/train/mbeir_train_cirr_task7_qrels.txt",
qrels_train_file_list = []
for dataset,datatset_task_name in dataset_to_val_data_file_middle_name_map.items():
    for task_name in datatset_task_name:
        qrels_train_file_list.append(os.path.join(Path_qrels_train, f"mbeir_train_{task_name}_qrels.txt"))
print("qrels_train_file_list[0]:", qrels_train_file_list[0])
print(len(qrels_train_file_list))
print(qrels_train_file_list)

In [ ]:
old_qrels_file_list = [
    "mbeir_cirr_train_qrels.txt",
    "mbeir_edis_train_qrels.txt",
    "mbeir_fashion200k_train_qrels.txt",
    "mbeir_fashioniq_train_qrels.txt",
    "mbeir_infoseek_train_qrels.txt",
    "mbeir_mscoco_train_qrels.txt",
    "mbeir_nights_train_qrels.txt",
    "mbeir_oven_train_qrels.txt",
    "mbeir_visualnews_train_qrels.txt",
    "mbeir_webqa_train_qrels.txt"
]
Path_qrels_train_old = "/home/user_name/U-MARVEL/data/M-BEIR/qrels/train"
old_qrels_file_list = [os.path.join(Path_qrels_train_old, path) for path in old_qrels_file_list]

In [ ]:
import shutil
# shutil.copy2(source_file, new_file_path)
for qrels_train_file in qrels_train_file_list:
    dataset = qrels_train_file.split("/")[-1].split("_")[2]
    for old_qrels_file in old_qrels_file_list:
        if dataset in old_qrels_file:
            print(qrels_train_file,old_qrels_file)
            # 复制文件
            shutil.copy2(old_qrels_file, qrels_train_file)
            break

### 根据 qid 对训练集进行去重

In [ ]:
import os
import sys
import json
from tqdm import tqdm
from collections import defaultdict
Path_query_train = "/home/user_name/U-MARVEL/data/M-BEIR/query/train/query_train"
# 任务映射
dataset_to_val_data_file_middle_name_map = {
            "VisualNews": ["visualnews_task0", "visualnews_task3"],
            "MSCOCO": ["mscoco_task0", "mscoco_task3"],
            "Fashion200K": ["fashion200k_task0", "fashion200k_task3"],
            "WebQA": ["webqa_task1", "webqa_task2"],
            "EDIS": ["edis_task2"],
            "NIGHTS": ["nights_task4"],
            "OVEN": ["oven_task6", "oven_task8"],
            "INFOSEEK": ["infoseek_task6", "infoseek_task8"],
            "FashionIQ": ["fashioniq_task7"],
            "CIRR": ["cirr_task7"],
        }

# Mapping of dataset names to IDs
DATASET_IDS = {
    "visualnews": 0,     "fashion200k": 1,   "webqa": 2,      "edis": 3,
    "nights": 4,         "oven": 5,          "infoseek": 6,
    "fashioniq": 7,      "cirr": 8,          "mscoco": 9,
}

MBEIR_TASK = {
    "text -> image": 0,      "text -> text": 1,             "text -> image,text": 2,
    "image -> text": 3,      "image -> image": 4,           "image -> text,image": 5,
    "image,text -> text": 6, "image,text -> image": 7,      "image,text -> image,text": 8,
}
# "./data/M-BEIR/query/train/mbeir_train_visualnews_task3.jsonl",
query_train_file_list = []
for dataset,datatset_task_name in dataset_to_val_data_file_middle_name_map.items():
    for task_name in datatset_task_name:
        query_train_file_list.append(os.path.join(Path_query_train, f"mbeir_train_{task_name}.jsonl"))
print("query_train_file_list[0]:", query_train_file_list[0])
print(len(query_train_file_list))

In [ ]:
# 对 query_train_file_list 按照 qid 进行去重, 重新进行保存
num_1 = 0
num_2 = 0
for query_train_file in query_train_file_list:
    qid_set = set()
    data = defaultdict(dict)
    
    with open(query_train_file, 'r', encoding='utf-8') as f:
        print(query_train_file)
        for line in tqdm(f):
            item = json.loads(line)
            qid  = item['qid']
            num_2+=1
            if qid not in qid_set:
                qid_set.add(qid)
                data[qid] = item.copy()
            else:
                try: 
                    assert data[qid] == item
                except:
                    print(data[qid])
                    print(item)
    num_1 += len(data)
    new_file = query_train_file.replace('.jsonl', '_dedup.jsonl')
    with open(new_file, 'w', encoding='utf-8') as f:
        for item in data.values():
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f"未去重 Total number of qid in query train list: {num_2}")
print(f"去重 Total number of qid in query train list: {num_1}")

In [ ]:
# 对 query_train_file_list 按照 qid 进行去重, 重新进行保存
num_1 = 0
num_2 = 0
for query_train_file in query_train_file_list:
    new_file = query_train_file.replace('.jsonl', '_dedup.jsonl')
    qid_set = set()
    data = defaultdict(dict) 
    with open(new_file, 'r', encoding='utf-8') as f:
        print(new_file)
        for line in tqdm(f):
            item = json.loads(line)
            qid  = item['qid']
            num_2+=1
            if qid not in qid_set:
                qid_set.add(qid)
                data[qid] = item.copy()
            else:
                try: 
                    assert data[qid] == item
                except:
                    print(data[qid])
                    print(item)
    num_1 += len(data)
print(f"未去重 Total number of qid in query train list: {num_2}")
print(f"去重 Total number of qid in query train list: {num_1}")